# UIT_SGS_Foodseekers_C1 — XLM-RoBERTa (Lượt 3) · bản PyTorch thuần
**Mô hình (approved list):** `FacebookAI/xlm-roberta-base`

Phân loại 3 lớp `BENIGN / JAILBREAK / PROMPT_INJECTION` (pt/fr/it/en), chấm Macro F1.

Bản này **không dùng Trainer/TrainingArguments** nên chạy được trên mọi phiên bản transformers của Colab.
Cách chạy: **Runtime → Change runtime type → T4 GPU**, rồi **Runtime → Run all**.

In [ ]:
# 1. Cài & kiểm tra GPU
!pip -q install -U transformers accelerate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO GPU - hay bat T4!")

In [ ]:
# 2. Cấu hình
MODEL_NAME = "FacebookAI/xlm-roberta-base"
MAX_LEN, EPOCHS, LR, BATCH, SEED = 256, 3, 2e-5, 16, 42
BASE_URL = "https://raw.githubusercontent.com/bersious/Foodseekers/main/data"
LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
L2I = {l: i for i, l in enumerate(LABELS)}

import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 3. Tải dữ liệu từ repo đội
import pandas as pd
train = pd.read_csv(f"{BASE_URL}/train.csv")
test  = pd.read_csv(f"{BASE_URL}/test.csv")
train["text"] = train["text"].fillna("").astype(str)
test["text"]  = test["text"].fillna("").astype(str)
print(train.shape, test.shape)
print(train["label"].value_counts(normalize=True).round(3).to_dict())

In [ ]:
# 4. Chia train/validation giữ tỉ lệ nhãn x ngôn ngữ
from sklearn.model_selection import train_test_split
strat = train["label"] + "_" + train["language"]
tr_df, va_df = train_test_split(train, test_size=0.1, stratify=strat, random_state=SEED)
y_tr = tr_df["label"].map(L2I).values
y_va = va_df["label"].map(L2I).values
print(len(tr_df), len(va_df))

In [ ]:
# 5. Dataset + DataLoader (padding động theo batch)
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = labels
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        return (self.texts[i], -1 if self.labels is None else int(self.labels[i]))

def collate(batch):
    texts, labels = zip(*batch)
    enc = tok(list(texts), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(labels)
    return enc

dl_tr = DataLoader(DS(tr_df["text"], y_tr), batch_size=BATCH, shuffle=True, collate_fn=collate)
dl_va = DataLoader(DS(va_df["text"], y_va), batch_size=BATCH * 2, shuffle=False, collate_fn=collate)
dl_te = DataLoader(DS(test["text"]), batch_size=BATCH * 2, shuffle=False, collate_fn=collate)

In [ ]:
# 6. Model + loss có trọng số lớp (chống mất cân bằng 62/31/7)
import torch.nn as nn
from transformers import AutoModelForSequenceClassification

counts = np.bincount(y_tr, minlength=3)
class_w = torch.tensor(len(y_tr) / (3 * counts), dtype=torch.float).to(DEVICE)
print("Class weights:", class_w.cpu().numpy().round(3))

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label={i: l for i, l in enumerate(LABELS)}, label2id=L2I).to(DEVICE)

loss_fn = nn.CrossEntropyLoss(weight=class_w)
optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(dl_tr) * EPOCHS
try:
    from transformers import get_linear_schedule_with_warmup
    sched = get_linear_schedule_with_warmup(optim, int(0.1 * total_steps), total_steps)
except Exception:
    sched = None
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

In [ ]:
# 7. Vòng lặp huấn luyện + đánh giá mỗi epoch (giữ model tốt nhất theo Macro F1)
from sklearn.metrics import f1_score
from scipy.special import softmax
import copy

@torch.no_grad()
def predict(dl):
    model.eval(); logits = []
    for enc in dl:
        labels = enc.pop("labels")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            out = model(**enc)
        logits.append(out.logits.float().cpu().numpy())
    return np.concatenate(logits)

best_f1, best_state = -1, None
for ep in range(EPOCHS):
    model.train(); running = 0.0
    for step, enc in enumerate(dl_tr):
        labels = enc.pop("labels").to(DEVICE)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        optim.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(**enc).logits
            loss = loss_fn(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optim); scaler.update()
        if sched: sched.step()
        running += loss.item()
        if step % 100 == 0:
            print(f"  epoch {ep+1} step {step}/{len(dl_tr)} loss {running/(step+1):.4f}")
    va_f1 = f1_score(y_va, predict(dl_va).argmax(1), average="macro")
    print(f"[Epoch {ep+1}] Macro F1 (val, argmax) = {va_f1:.4f}")
    if va_f1 > best_f1:
        best_f1 = va_f1; best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
print("Best val Macro F1 (argmax):", round(best_f1, 4))

In [ ]:
# 8. Tinh chỉnh hệ số nhân xác suất lớp hiếm trên validation
from sklearn.metrics import classification_report
va_prob = softmax(predict(dl_va), axis=1)
best_w, best_tuned = (1.0, 1.0), -1
for wj in np.arange(0.8, 1.31, 0.05):
    for wp in np.arange(0.8, 2.01, 0.05):
        f = f1_score(y_va, (va_prob * [1.0, wj, wp]).argmax(1), average="macro")
        if f > best_tuned: best_tuned, best_w = f, (round(float(wj),2), round(float(wp),2))
print(f"He so JAILBREAK x{best_w[0]}, PROMPT_INJECTION x{best_w[1]} -> Macro F1 = {best_tuned:.4f}")
print(classification_report(y_va, (va_prob * [1.0, *best_w]).argmax(1), target_names=LABELS, digits=4))
for lang in ["pt", "fr", "it", "en"]:
    m = (va_df["language"] == lang).values
    print(lang, round(f1_score(y_va[m], (va_prob[m]*[1.0,*best_w]).argmax(1), average="macro"), 4))

In [ ]:
# 9. Dự đoán test + kiểm tra toàn vẹn + lưu file
te_prob = softmax(predict(dl_te), axis=1)
pred = (te_prob * [1.0, *best_w]).argmax(1)
sub = pd.DataFrame({"id": test["id"], "label": [LABELS[i] for i in pred]})
assert len(sub) == 6000 and sub["label"].isin(LABELS).all() and sub["label"].notna().all() and sub["id"].is_unique
sub.to_csv("submission.csv", index=False)
np.save("xlmr_test_proba.npy", te_prob)
print(sub["label"].value_counts().to_dict())

In [ ]:
# 10. Tải file về máy
try:
    from google.colab import files
    files.download("submission.csv"); files.download("xlmr_test_proba.npy")
except Exception:
    print("File nam o thu muc hien tai.")